### Tool Integration in Langchain
1. How to create tools
2. how to use built-in tools and toolkits
3. how to use chat models to call tools
4. how to pass tool outputs to chat models

This @tool decorator is the simplest way to define a custom tool. The decorator uses the function name as the tool name by default, but this can be overridden by passing a string as the first argument. Additionally, the decorator will use the function's docstring as the tool's description - so a docstring MUST be provided.

In [1]:
## @tool decorator

from langchain_core.tools import tool

@tool
def division(a:int,b:int)->int:
    """Divide 2 number"""
    return a/b

In [2]:
print(division.name)
print(division.description)
print(division.args)

division
Divide 2 number
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [3]:
## async implementation ---useful to keep running program without stopping or pausing it

from langchain_core.tools import tool


@tool
async def amultiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

In [4]:
from typing import Annotated,List

@tool
def multiply_by_max(
    a: Annotated[int,"A value"],
    b:Annotated[List[int],"list of ints over which to take maximum"]
)->int:
    """Mulitply a by the maximum of b"""
    return a* max(b)


In [5]:
print(multiply_by_max.args)

{'a': {'description': 'A value', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'list of ints over which to take maximum', 'items': {'type': 'integer'}, 'title': 'B', 'type': 'array'}}


### Structured tool
The StructuredTool.from_function class method provides a bit more configurability than the @tool decorator, without requiring much additional code.

In [6]:
from langchain_core.tools import StructuredTool


def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b


async def amultiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

calculator=StructuredTool.from_function(func=multiply,coroutine=amultiply)

print(calculator.invoke({"a": 2, "b": 3}))
print(await calculator.ainvoke({"a": 2, "b": 5}))


6
10


### InBuilt tools
### Wikipedia Integration


In [9]:
# user agent request to wikipedia showing the credibility of the user
import wikipedia
wikipedia.set_user_agent("AgenticAILearning/1.0 (your-email@example.com)")

from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

api_wrapper = WikipediaAPIWrapper(top_k_results=5, doc_content_chars_max=500)
tool = WikipediaQueryRun(api_wrapper=api_wrapper)

print(tool.invoke({"query": "langchain"}))

Page: LangChain
Summary: LangChain is a software framework that helps facilitate the integration of large language models (LLMs) into applications. As a language model integration framework, LangChain's use-cases largely overlap with those of language models in general, including document analysis and summarization, chatbots, and code analysis.

Page: List of artificial intelligence companies
Summary: Below is a list of notable companies that primarily focus on artificial intelligence (AI). Comp


In [10]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

api_wrapper=WikipediaAPIWrapper(top_k_results=5,doc_content_chars_max=500)
tool=WikipediaQueryRun(api_wrapper=api_wrapper)

print(tool.invoke({"query":"langchain"}))

Page: LangChain
Summary: LangChain is a software framework that helps facilitate the integration of large language models (LLMs) into applications. As a language model integration framework, LangChain's use-cases largely overlap with those of language models in general, including document analysis and summarization, chatbots, and code analysis.

Page: List of artificial intelligence companies
Summary: Below is a list of notable companies that primarily focus on artificial intelligence (AI). Comp


In [16]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [17]:
os.environ["TAVILY_API_KEY"]=os.getenv("TAVILY_API_KEY")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [19]:
from langchain_tavily import TavilySearch

tool = TavilySearch(
    max_results=5,
    topic="general",
    # include_answer=False,
    # include_raw_content=False,
    # include_images=False,
    # include_image_descriptions=False,
    # search_depth="basic",
    # time_range="day",
    # include_domains=None,
    # exclude_domains=None
)

In [20]:
tool.invoke("What is the recent ai news")

{'query': 'What is the recent ai news',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.reddit.com/r/AISEOInsider/comments/1rs1o1v/latest_ai_news_the_biggest_ai_announcements_right',
   'title': 'The Biggest AI Announcements Right Now : r/AISEOInsider',
   'content': 'More builders discovering the latest AI news are also joining the AI Profit Boardroom to learn practical ways to use these tools for automation and AI-powered workflows.\n\n# Frequently Asked Questions About Latest AI News\n\n1.   What Does Latest AI News Usually Cover? Latest AI news includes new AI tools, company announcements, research breakthroughs, and startup funding. [...] 2.   Which AI Tools Appeared In Latest AI News Recently? Recent tools include Gemini Workspace AI, Nvidia Nemo Claw, Claude Code Review, Pomelli marketing AI, and Hermes AI agent.\n\n3.   Why Are AI Agents Important In Latest AI News? AI agents can perform tasks automatically across applications whi

In [23]:
from langchain_community.utilities import ArxivAPIWrapper

tool=ArxivAPIWrapper()

import arxiv
from langchain_core.tools import tool

client = arxiv.Client()

@tool
def arxiv_search(query: str) -> str:
    """Search arXiv for research papers by topic or arXiv ID (e.g. 1706.03762)."""
    if query.replace(".", "").replace("v", "").isdigit():
        search = arxiv.Search(id_list=[query])
    else:
        search = arxiv.Search(query=query, max_results=3)

    papers = []
    for r in client.results(search):
        papers.append(
            f"Title: {r.title}\n"
            f"Published: {r.published.date()}\n"
            f"Authors: {', '.join(a.name for a in r.authors)}\n"
            f"Summary: {r.summary[:500]}"
        )
    return "\n\n".join(papers) or "No papers found."

print(arxiv_search.invoke({"query": "1706.03762"}))

Title: Attention Is All You Need
Published: 2017-06-12
Authors: Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Lukasz Kaiser, Illia Polosukhin
Summary: The dominant sequence transduction models are based on complex recurrent or convolutional neural networks in an encoder-decoder configuration. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. Experiments on two machine translation tasks show these models to be superior in quality while being more 


### Call tool with LLM Model


In [24]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=100)
wiki_tool = WikipediaQueryRun(api_wrapper=api_wrapper)

print(wiki_tool.invoke({"query": "langchain"}))

Page: LangChain
Summary: LangChain is a software framework that helps facilitate the integration of 


In [25]:
from langchain_tavily import TavilySearch

tavily_tool = TavilySearch(
    max_results=5,
    topic="general",
    # include_answer=False,
    # include_raw_content=False,
    # include_images=False,
    # include_image_descriptions=False,
    # search_depth="basic",
    # time_range="day",
    # include_domains=None,
    # exclude_domains=None
)

In [26]:
from langchain_core.tools import tool

@tool
def add(a: int, b: int) -> int:
    """Adds a and b."""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """Multiplies a and b."""
    return a * b

In [28]:
wiki_tool.name


'wikipedia'

In [29]:
tavily_tool.name

'tavily_search'

In [30]:
add.name

'add'

In [31]:
multiply.name

'multiply'

In [32]:
tools=[wiki_tool,add,multiply,tavily_tool]
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'd:\\Self Upskilling\\AI\\Agentic AI\\.venv\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=100)),
 StructuredTool(name='add', description='Adds a and b.', args_schema=<class 'langchain_core.utils.pydantic.add'>, func=<function add at 0x00000255311D9DA0>),
 StructuredTool(name='multiply', description='Multiplies a and b.', args_schema=<class 'langchain_core.utils.pydantic.multiply'>, func=<function multiply at 0x00000255311D9580>),
 TavilySearch(max_results=5, topic='general', api_wrapper=TavilySearchAPIWrapper(tavily_api_key=SecretStr('**********'), api_base_url=None))]

In [36]:
from langchain.chat_models import init_chat_model
llm=init_chat_model("qwen/qwen3.8-27b",model_provider="groq")
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.12', 'langchain': '1.4.6'}}, client=<groq.resources.chat.completions.Completions object at 0x00000255319A9D90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000255319A80E0>, model_name='qwen/qwen3.8-27b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [37]:
llm_with_tool=llm.bind_tools(tools)
llm_with_tool

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.12', 'langchain': '1.4.6'}}, client=<groq.resources.chat.completions.Completions object at 0x00000255319A9D90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000255319A80E0>, model_name='qwen/qwen3.8-27b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'wikipedia', 'description': 'A wrapper around Wikipedia. Useful for when you need to answer general questions about people, places, companies, facts, historical events, or other subjects. Input should be a search query.', 'parameters': {'properties': {'query': {'description': 'query to look up on wikipedia', 'type': 'string'}}, 'required': ['query'], 'type': 'object'}}}, {'type': 'function', 'function': {'name': 'add', 'description': 'Adds a and b.', 'parameters': {'properties': {'a': {'type': 'integer'}, 'b': {'type': 'integer'}}, 'required': ['a', 'b'], 'ty

In [38]:
from langchain_core.messages import HumanMessage
query="What is 2* 3"
messages = [HumanMessage(query)]
response=llm_with_tool.invoke(query)
print(response)

content='' additional_kwargs={'tool_calls': [{'id': 'nmg8gvwn0', 'function': {'arguments': '{"a":2,"b":3}', 'name': 'multiply'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 2062, 'total_tokens': 2098, 'completion_time': 0.082065023, 'completion_tokens_details': None, 'prompt_time': 0.153004733, 'prompt_tokens_details': None, 'queue_time': 0.086925421, 'total_time': 0.235069756}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_8999620a6e', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a1744e-82b2-71a1-945a-e3da24abf023-0' tool_calls=[{'name': 'multiply', 'args': {'a': 2, 'b': 3}, 'id': 'nmg8gvwn0', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 2062, 'output_tokens': 36, 'total_tokens': 2098}


In [39]:
response

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'nmg8gvwn0', 'function': {'arguments': '{"a":2,"b":3}', 'name': 'multiply'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 2062, 'total_tokens': 2098, 'completion_time': 0.082065023, 'completion_tokens_details': None, 'prompt_time': 0.153004733, 'prompt_tokens_details': None, 'queue_time': 0.086925421, 'total_time': 0.235069756}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_8999620a6e', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1744e-82b2-71a1-945a-e3da24abf023-0', tool_calls=[{'name': 'multiply', 'args': {'a': 2, 'b': 3}, 'id': 'nmg8gvwn0', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 2062, 'output_tokens': 36, 'total_tokens': 2098})

In [40]:
response.tool_calls

[{'name': 'multiply',
  'args': {'a': 2, 'b': 3},
  'id': 'nmg8gvwn0',
  'type': 'tool_call'}]

In [41]:
for tool_call in response.tool_calls:
    selected_tool = {"add": add, "multiply": multiply,"wikipedia":wiki_tool,"tavily_search":tavily_tool}[tool_call["name"].lower()]
    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_msg)
    
messages

[HumanMessage(content='What is 2* 3', additional_kwargs={}, response_metadata={}),
 ToolMessage(content='6', name='multiply', tool_call_id='nmg8gvwn0')]

In [42]:
messages

[HumanMessage(content='What is 2* 3', additional_kwargs={}, response_metadata={}),
 ToolMessage(content='6', name='multiply', tool_call_id='nmg8gvwn0')]

In [43]:
llm_with_tool.invoke(messages)

AIMessage(content='2 × 3 = **6**', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 2072, 'total_tokens': 2081, 'completion_time': 0.020327984, 'completion_tokens_details': None, 'prompt_time': 0.179076092, 'prompt_tokens_details': None, 'queue_time': 0.056646366, 'total_time': 0.199404076}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_43dd1ea4ee', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1744f-8bbe-7df0-bb94-1db1ac059b30-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 2072, 'output_tokens': 9, 'total_tokens': 2081})

In [44]:
from langchain_core.messages import HumanMessage

query = "What is langchain and what is 5*15?"
messages = [HumanMessage(query)]

ai_msg = llm_with_tool.invoke(messages)
ai_msg

AIMessage(content="LangChain is an open-source framework for developing applications powered by large language models (LLMs). It provides the tools and abstractions needed to build LLM-powered applications, including:\n\n- **Chains**: Composable sequences of LLM calls, where the output of one step feeds into the next.\n- **Retrieval-Augmented Generation (RAG)**: Integrating external data sources (like documents or databases) to improve LLM responses.\n- **Agents**: LLMs that can use tools, make decisions, and take actions.\n- **Integrations**: Support for many LLM providers (OpenAI, Anthropic, Hugging Face, etc.), vector stores, and other services.\n\nIt's written in Python (and also available in JavaScript/TypeScript) and is widely used for building chatbots, assistants, and AI-powered applications.\n\nAs for 5 × 15:\n\n", additional_kwargs={'tool_calls': [{'id': 'y92wx7ctt', 'function': {'arguments': '{"a":5,"b":15}', 'name': 'multiply'}, 'type': 'function'}]}, response_metadata={'to

In [45]:
ai_msg.tool_calls

[{'name': 'multiply',
  'args': {'a': 5, 'b': 15},
  'id': 'y92wx7ctt',
  'type': 'tool_call'}]

In [46]:
for tool_call in ai_msg.tool_calls:
    selected_tool = {"add": add, "multiply": multiply,"wikipedia":wiki_tool,"tavily_search":tavily_tool}[tool_call["name"].lower()]
    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_msg)
    
messages

[HumanMessage(content='What is langchain and what is 5*15?', additional_kwargs={}, response_metadata={}),
 ToolMessage(content='75', name='multiply', tool_call_id='y92wx7ctt')]

In [47]:
response=llm_with_tool.invoke(messages)

In [51]:
response.content
messages.append(response) 

In [52]:
from langchain_core.messages import HumanMessage

query = "What is langchain and what is 5*15 and summarize the recent AI news ?"
messages = [HumanMessage(query)]

ai_msg = llm_with_tool.invoke(messages)
ai_msg

AIMessage(content='Here are the answers to your three questions:\n\n### 1. What is LangChain?\n**LangChain** is a popular open-source framework designed for building applications powered by Large Language Models (LLMs). It provides developers with tools, abstractions, and integrations to create AI-powered applications such as chatbots, agents, and retrieval-augmented generation (RAG) systems. Key features include:\n*   **Modularity:** Composable components for chains, agents, and memory.\n*   **Integrations:** Supports numerous LLMs (OpenAI, Anthropic, Hugging Face, etc.), vector databases, and other tools.\n*   **LangChain Expressions (LCEL):** A streamlined interface for building and debugging chains.\n*   **LangGraph:** An extension for building stateful, multi-step agent workflows.\n\n### 2. What is 5 * 15?\n**5 × 15 = 75**\n\nLet me verify this with a calculation:\n\n', additional_kwargs={'tool_calls': [{'id': '3ctgt6jam', 'function': {'arguments': '{"a":5,"b":15}', 'name': 'multi

In [53]:
ai_msg.tool_calls

[{'name': 'multiply',
  'args': {'a': 5, 'b': 15},
  'id': '3ctgt6jam',
  'type': 'tool_call'}]

In [54]:
for tool_call in ai_msg.tool_calls:
    selected_tool = {"add": add, "multiply": multiply,"wikipedia":wiki_tool,"tavily_search":tavily_tool}[tool_call["name"].lower()]
    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_msg)
    
messages

[HumanMessage(content='What is langchain and what is 5*15 and summarize the recent AI news ?', additional_kwargs={}, response_metadata={}),
 ToolMessage(content='75', name='multiply', tool_call_id='3ctgt6jam')]

In [55]:
response=llm_with_tool.invoke(messages)
response.content

'Here is the information you requested:\n\n**1. What is LangChain?**\nLangChain is an open-source framework for developing applications powered by large language models (LLMs). It provides a set of tools and abstractions that make it easier to build context-aware reasoning applications, including chatbots, RAG (Retrieval-Augmented Generation) systems, and agents. Its key features include:\n*   **Modular Components:** It offers interchangeable components for prompt templates, output parsers, vector stores, and LLM providers.\n*   **Chaining:** It allows developers to "chain" together multiple LLM calls and other operations to create complex workflows.\n*   **Agent Support:** It enables the creation of AI agents that can use tools (like search engines or code interpreters) to solve tasks autonomously.\n*   **Integration:** It has extensive integrations with various LLMs (OpenAI, Anthropic, etc.), vector databases, and external APIs.\n\n**2. What is 5 * 15?**\nAs calculated above, **5 * 1